# 🤖 Fase 3 — Análisis IA con Qwen3-8B
**Pipeline DIGI Spain Telecom — Control de Calidad C2C**

**Nodos:** 10 (LLM Qwen3-8B) · 11 (Cálculo de puntuaciones + RAG prompt)

**Entrada:** `guion_diarizado.txt` (salida de Fase 2)  
**Salida:** `resultado_fase3.json` (entrada de Fase 4 — PDF)  

**Colab:** Qwen3-8B Q4 (bitsandbytes 4-bit) en T4 (16GB VRAM)  
**Docker:** Qwen3-235B vía vLLM

---

## ⚠️ PASO 0 — Auto-fix NumPy (ejecutar siempre primero)

In [ ]:
import subprocess, sys, importlib.util

spec = importlib.util.find_spec('numpy')
if spec:
    import numpy as _np
    version = tuple(int(x) for x in _np.__version__.split('.')[:2])
    if version >= (2, 0):
        print(f'⚠️  NumPy {_np.__version__} detectado — incompatible con bitsandbytes')
        subprocess.check_call([
            sys.executable, '-m', 'pip', 'install',
            'numpy==1.26.4',
            '--target', '/usr/local/lib/python3.12/dist-packages/',
            '--no-deps', '-q'
        ])
        print('🔴 ACCIÓN REQUERIDA: Runtime → Restart session y vuelve a ejecutar desde aquí')
    else:
        print(f'✅ NumPy {_np.__version__} — compatible, continúa')
else:
    print('ℹ️  NumPy no instalado aún — continúa')

## PASO 1 — Instalar dependencias

In [ ]:
# Transformers + bitsandbytes para Qwen3-8B en 4-bit
!pip install -q transformers>=4.44.0 bitsandbytes>=0.43.0 accelerate>=0.31.0
!pip install -q sentencepiece protobuf

# Verificar GPU
import torch
if torch.cuda.is_available():
    gpu = torch.cuda.get_device_properties(0)
    vram_gb = gpu.total_memory / 1e9
    print(f'✅ GPU: {gpu.name} — {vram_gb:.1f} GB VRAM')
    if vram_gb < 10:
        print('⚠️  VRAM baja — Qwen3-8B Q4 necesita ~6 GB mínimo')
    else:
        print('✅ VRAM suficiente para Qwen3-8B Q4')
else:
    print('❌ No hay GPU — activa Runtime → T4 GPU')

## PASO 2 — Configuración

In [ ]:
import json
from datetime import date
from pathlib import Path
from google.colab import drive

# --- Montar Drive ---
drive.mount('/content/drive')

# ================================================================
#  CONFIGURACIÓN DE RUTAS
#  El sistema busca guion_diarizado.txt automáticamente en Drive.
#  Si lo encuentra en varios sitios, elige el más reciente.
#  Puedes forzar una ruta manual descomentando RUTA_MANUAL.
# ================================================================

RUTA_MANUAL = None  # Ejemplo: '/content/drive/MyDrive/mis_archivos/guion_diarizado.txt'

def buscar_guion(drive_root='/content/drive/MyDrive'):
    """Busca guion_diarizado.txt en todo el Drive, devuelve el más reciente."""
    candidatos = list(Path(drive_root).rglob('guion_diarizado.txt'))
    if not candidatos:
        return None
    # Ordenar por fecha de modificación, más reciente primero
    candidatos.sort(key=lambda p: p.stat().st_mtime, reverse=True)
    return candidatos[0]

# Resolver ruta del guión
if RUTA_MANUAL:
    GUION_PATH = Path(RUTA_MANUAL)
    print(f'📌 Usando ruta manual: {GUION_PATH}')
else:
    print('🔍 Buscando guion_diarizado.txt en Google Drive...')
    GUION_PATH = buscar_guion()

if GUION_PATH and GUION_PATH.exists():
    print(f'✅ Guión encontrado: {GUION_PATH}')
    print(f'   Tamaño: {GUION_PATH.stat().st_size:,} bytes')
    # La carpeta de salida es la misma donde está el guión
    RESULTADO_PATH = GUION_PATH.parent / 'resultado_fase3.json'
    print(f'   Salida: {RESULTADO_PATH}')
else:
    print('❌ No se encontró guion_diarizado.txt en Google Drive.')
    print()
    print('   Opciones:')
    print('   1. Ejecuta el notebook de Fase 2 primero para generar el guión')
    print('   2. Sube el archivo guion_diarizado.txt a tu Drive manualmente')
    print('   3. Pon la ruta exacta en RUTA_MANUAL al inicio de esta celda')
    raise FileNotFoundError('guion_diarizado.txt no encontrado')

# Metadatos de la llamada
META = {
    'id_grabacion': '',           # Vacío → se genera automáticamente
    'asesor': 'Gabriel Barreto',  # Nombre del asesor
    'fecha_llamada': '',          # YYYY-MM-DD o vacío
    'servicio': 'C2C',
    'fecha_auditoria': str(date.today()),
}

## PASO 3 — Cargar guión diarizado

In [ ]:
# Leer el guión completo
transcripcion_raw = GUION_PATH.read_text(encoding='utf-8')

print('📄 Guión cargado:')
print('=' * 60)
print(transcripcion_raw[:3000])  # Preview de los primeros 3000 chars
if len(transcripcion_raw) > 3000:
    print(f'... [{len(transcripcion_raw) - 3000} caracteres más]')
print('=' * 60)
print(f'\nTotal: {len(transcripcion_raw)} caracteres')

## NODO 11 — Rúbrica DIGI (base de conocimiento)

> **Arquitectura:** En Colab inyectamos la rúbrica completa en el prompt (RAG implícito).  
> En Docker, este nodo usa BGE-M3 + pgvector para recuperación semántica sobre toda la documentación de DIGI.

In [ ]:
# ============================================================
#  RÚBRICA DIGI — 30 criterios · 2 secciones
#  Pesos base: escala libre (normalización dinámica al calcular)
#  * = criterio crítico (incumplimiento implica alerta especial)
# ============================================================

RUBRICA = [
    # ─── SECCIÓN 1: ESTRUCTURA DE LA LLAMADA ───────────────────────
    {
        'seccion': 'Estructura de la llamada',
        'subseccion': 'Presentación',
        'id': 'p1',
        'pregunta': 'El asesor hace un uso correcto del saludo corporativo al inicio de la llamada',
        'guia': 'El asesor debe saludar indicando el nombre del servicio (DIGI) y el suyo propio de forma clara y amable.',
        'critico': False,
        'peso': 1.0,
    },
    {
        'seccion': 'Estructura de la llamada',
        'subseccion': 'Política de seguridad',
        'id': 'ps1',
        'pregunta': 'Identificación Correcta/Completa en política de seguridad/datos *',
        'guia': 'El asesor debe verificar la identidad del cliente (nombre completo, DNI u otro dato de seguridad). Marcador crítico: si no se identifica al cliente la llamada puede tener consecuencias legales.',
        'critico': True,
        'peso': 1.0,
    },
    {
        'seccion': 'Estructura de la llamada',
        'subseccion': 'Política de seguridad',
        'id': 'ps2',
        'pregunta': 'Realiza las verificaciones previas necesarias a la contratación (comprobar paquetes/pedidos/portabilidades, máx. líneas)',
        'guia': 'Antes de ofrecer nada, el asesor revisa en sistemas si el cliente ya tiene pedidos activos, portabilidades en curso o ha alcanzado el límite de líneas. N/A si la llamada no es de contratación nueva.',
        'critico': False,
        'peso': 2.0,
    },
    {
        'seccion': 'Estructura de la llamada',
        'subseccion': 'Sondeo',
        'id': 's1',
        'pregunta': 'Averigua la necesidad de su cliente mediante preguntas de sondeo',
        'guia': 'El asesor hace preguntas abiertas para entender qué busca el cliente (más datos, fibra, precio, más líneas, etc.). N/A si el cliente llama con una necesidad muy específica ya expresada.',
        'critico': False,
        'peso': 3.0,
    },
    {
        'seccion': 'Estructura de la llamada',
        'subseccion': 'Sondeo',
        'id': 's2',
        'pregunta': 'Conoce los hábitos de consumo del cliente con preguntas de sondeo',
        'guia': 'El asesor pregunta sobre el uso del cliente: cuántas líneas tiene, si usa datos, si tiene fibra en casa, cuántos GB consume, etc.',
        'critico': False,
        'peso': 3.0,
    },
    {
        'seccion': 'Estructura de la llamada',
        'subseccion': 'Sondeo',
        'id': 's3',
        'pregunta': 'Ha sido capaz de conocer lo que paga el cliente en su actual operador',
        'guia': 'El asesor pregunta directamente cuánto paga el cliente con su operador actual para poder comparar con la oferta DIGI. N/A si el cliente ya es cliente DIGI o no tiene operador previo.',
        'critico': False,
        'peso': 3.0,
    },
    {
        'seccion': 'Estructura de la llamada',
        'subseccion': 'Asignación de oferta',
        'id': 'ao1',
        'pregunta': 'Consulta de cobertura',
        'guia': 'El asesor verifica que hay cobertura de fibra o móvil DIGI en la dirección del cliente antes de ofrecer el producto. N/A si la llamada no implica contratación de fibra.',
        'critico': False,
        'peso': 2.5,
    },
    {
        'seccion': 'Estructura de la llamada',
        'subseccion': 'Asignación de oferta',
        'id': 'ao2',
        'pregunta': 'El agente orienta al cliente hacia venta valor y máximos productos (VENTA CRUZADA)',
        'guia': 'El asesor intenta ofrecer productos adicionales (fibra + móvil, líneas adicionales, TV) para maximizar el ticket. No es N/A si hay oportunidad de venta cruzada aunque el cliente no la pida.',
        'critico': False,
        'peso': 3.0,
    },
    {
        'seccion': 'Estructura de la llamada',
        'subseccion': 'Asignación de oferta',
        'id': 'ao3',
        'pregunta': 'Información correcta y completa sobre producto y permanencia',
        'guia': 'El asesor explica claramente el precio del producto, las condiciones de permanencia, los servicios incluidos y los que no. Debe ser información veraz y completa.',
        'critico': False,
        'peso': 4.0,
    },
    {
        'seccion': 'Estructura de la llamada',
        'subseccion': 'Asignación de oferta',
        'id': 'ao4',
        'pregunta': 'Pone en valor las características estrella de DIGI (precio final, cobertura Movistar, acumular MB, etc.)',
        'guia': 'El asesor menciona y destaca activamente las ventajas clave de DIGI: precio competitivo, red Movistar, acumulación de datos no usados, sin letra pequeña, etc.',
        'critico': False,
        'peso': 2.0,
    },
    {
        'seccion': 'Estructura de la llamada',
        'subseccion': 'Asignación de oferta',
        'id': 'ao5',
        'pregunta': 'Cuantifica el ahorro mensual/anual haciendo ver al cliente el beneficio económico',
        'guia': 'El asesor calcula y menciona cuánto dinero ahorrará el cliente al mes o al año respecto a su operador actual. N/A si no se conoce el precio del operador actual.',
        'critico': False,
        'peso': 3.0,
    },
    {
        'seccion': 'Estructura de la llamada',
        'subseccion': 'Cierre de la venta',
        'id': 'cv1',
        'pregunta': 'El asesor orienta el cierre de la venta en la primera llamada, evitando agendar para otra',
        'guia': 'El asesor trabaja para cerrar la contratación en esta misma llamada y no propone llamar otro día salvo causa justificada. Es un indicador clave de productividad.',
        'critico': False,
        'peso': 9.0,
    },
    {
        'seccion': 'Estructura de la llamada',
        'subseccion': 'Cierre de la venta',
        'id': 'cv2',
        'pregunta': 'Rebate objeciones de manera correcta',
        'guia': 'Cuando el cliente pone objeciones (precio, permanencia, cobertura, confianza), el asesor responde con argumentos sólidos y verazes para superarlas. N/A si el cliente no plantea ninguna objeción.',
        'critico': False,
        'peso': 7.5,
    },
    {
        'seccion': 'Estructura de la llamada',
        'subseccion': 'Cierre de la venta',
        'id': 'cv3',
        'pregunta': 'Realiza un cierre directo de la venta',
        'guia': 'El asesor hace una pregunta de cierre directa y explícita ("¿Empezamos con la contratación?", "¿Le parece bien proceder?"). No espera a que el cliente proponga cerrar.',
        'critico': False,
        'peso': 8.0,
    },
    {
        'seccion': 'Estructura de la llamada',
        'subseccion': 'Toma de datos y registro en sistemas',
        'id': 'td1',
        'pregunta': 'El agente reformula los datos críticos (móvil de contacto, titular en origen, DNI, email) para evitar rechazos',
        'guia': 'El asesor repite en voz alta los datos proporcionados por el cliente para confirmarlos, reduciendo errores de contratación. N/A si no se realiza contratación.',
        'critico': False,
        'peso': 2.0,
    },
    {
        'seccion': 'Estructura de la llamada',
        'subseccion': 'Toma de datos y registro en sistemas',
        'id': 'td2',
        'pregunta': 'El asesor codifica la llamada de forma correcta y completa en el CRM',
        'guia': 'El asesor registra el resultado de la llamada en el CRM con la tipificación adecuada. Se evalúa si menciona que va a hacerlo o si hay evidencia en el audio. N/A si el asesor trabaja sin CRM en esta llamada.',
        'critico': False,
        'peso': 2.0,
    },
    {
        'seccion': 'Estructura de la llamada',
        'subseccion': 'Toma de datos y registro en sistemas',
        'id': 'td3',
        'pregunta': 'El asesor tipifica la llamada correctamente en Mediatel *',
        'guia': 'Mediatel es el sistema de tipificación de llamadas de DIGI. El asesor debe tipificar según el resultado (venta, información, reclamación, etc.). Crítico por su impacto en métricas.',
        'critico': True,
        'peso': 1.5,
    },
    {
        'seccion': 'Estructura de la llamada',
        'subseccion': 'Cierre de la llamada',
        'id': 'cl1',
        'pregunta': 'El asesor reformula los acuerdos alcanzados en la oferta comercial',
        'guia': 'Antes de despedirse, el asesor resume lo acordado: producto contratado, precio, plazos de instalación/portabilidad, próximas acciones.',
        'critico': False,
        'peso': 2.0,
    },
    {
        'seccion': 'Estructura de la llamada',
        'subseccion': 'Cierre de la llamada',
        'id': 'cl2',
        'pregunta': 'El asesor informa de los próximos pasos del proceso (plazos de instalación/portabilidad)',
        'guia': 'El asesor indica al cliente qué pasará después: cuándo recibirá el router, cuándo se hará efectiva la portabilidad, si necesita estar en casa para el técnico, etc.',
        'critico': False,
        'peso': 1.5,
    },
    {
        'seccion': 'Estructura de la llamada',
        'subseccion': 'Cierre de la llamada',
        'id': 'cl3',
        'pregunta': 'El asesor realiza despedida corporativa',
        'guia': 'La llamada termina con una despedida amable y corporativa que mencione DIGI, agradezca al cliente y desee un buen día.',
        'critico': False,
        'peso': 1.0,
    },
    # ─── SECCIÓN 2: ACTITUD / COMUNICACIÓN ─────────────────────────
    {
        'seccion': 'Actitud / Comunicación',
        'subseccion': 'Trato al cliente',
        'id': 'tc1',
        'pregunta': 'Personaliza la llamada',
        'guia': 'El asesor usa el nombre del cliente a lo largo de la llamada y adapta su discurso a la situación particular del cliente.',
        'critico': False,
        'peso': 1.0,
    },
    {
        'seccion': 'Actitud / Comunicación',
        'subseccion': 'Trato al cliente',
        'id': 'tc2',
        'pregunta': 'Gestión y ayuda en el proceso de firma',
        'guia': 'El asesor guía al cliente paso a paso durante el proceso de firma digital o confirmación de contrato, resolviendo sus dudas. N/A si no hay firma en esta llamada.',
        'critico': False,
        'peso': 1.5,
    },
    {
        'seccion': 'Actitud / Comunicación',
        'subseccion': 'Trato al cliente',
        'id': 'tc3',
        'pregunta': 'Muestra actitud empática y cercana con el cliente. Trato amable y cortés',
        'guia': 'El asesor demuestra empatía, escucha activamente, no interrumpe innecesariamente y mantiene un tono cálido y profesional durante toda la llamada.',
        'critico': False,
        'peso': 2.5,
    },
    {
        'seccion': 'Actitud / Comunicación',
        'subseccion': 'Trato al cliente',
        'id': 'tc4',
        'pregunta': 'Evita el maltrato al cliente e información no veraz *',
        'guia': 'El asesor NO engaña al cliente, NO usa tácticas de presión abusivas, NO da información falsa sobre precios o productos. Crítico: una sola infracción aquí puede anular la auditoría.',
        'critico': True,
        'peso': 3.5,
    },
    {
        'seccion': 'Actitud / Comunicación',
        'subseccion': 'Gestión de esperas',
        'id': 'ge1',
        'pregunta': 'Realiza una adecuada gestión de los tiempos de espera',
        'guia': 'Cuando el asesor necesita poner al cliente en espera, lo avisa, pide permiso, especifica el tiempo aproximado y retoma la llamada saludando. N/A si no hay esperas en la llamada.',
        'critico': False,
        'peso': 2.5,
    },
    {
        'seccion': 'Actitud / Comunicación',
        'subseccion': 'Gestión de esperas',
        'id': 'ge2',
        'pregunta': 'Estructura correctamente la llamada (acogida, sondeo, oferta, cierre) evitando dilatarla',
        'guia': 'La llamada sigue un flujo lógico y eficiente: presentación → sondeo → oferta → cierre. El asesor no divaga ni alarga innecesariamente la llamada.',
        'critico': False,
        'peso': 3.0,
    },
    {
        'seccion': 'Actitud / Comunicación',
        'subseccion': 'Capacidad de comunicación',
        'id': 'cc1',
        'pregunta': 'Vocabulario correcto, adaptado al cliente y evitando el uso de muletillas',
        'guia': 'El asesor habla con claridad, sin muletillas excesivas ("o sea", "este...", "básicamente"), adapta el nivel técnico al cliente y evita jerga interna.',
        'critico': False,
        'peso': 2.5,
    },
    {
        'seccion': 'Actitud / Comunicación',
        'subseccion': 'Capacidad de comunicación',
        'id': 'cc2',
        'pregunta': 'Transmite una imagen positiva de la marca y productos DIGI *',
        'guia': 'El asesor habla de DIGI de forma positiva, con convicción y entusiasmo. No hace comentarios negativos sobre los productos, la empresa o la competencia de forma inapropiada.',
        'critico': True,
        'peso': 2.5,
    },
    {
        'seccion': 'Actitud / Comunicación',
        'subseccion': 'Capacidad de comunicación',
        'id': 'cc3',
        'pregunta': 'Dirige la llamada evitando interrumpir al cliente',
        'guia': 'El asesor conduce la llamada hacia los objetivos sin cortar las frases del cliente. Deja hablar, escucha y retoma el control de forma natural.',
        'critico': False,
        'peso': 2.5,
    },
    {
        'seccion': 'Actitud / Comunicación',
        'subseccion': 'Capacidad de comunicación',
        'id': 'cc4',
        'pregunta': 'Aplica el procedimiento correcto',
        'guia': 'El asesor sigue los procedimientos estándar de DIGI para el tipo de llamada (contratación nueva, portabilidad, incidencia, etc.). No improvisa de forma que pueda generar problemas.',
        'critico': False,
        'peso': 2.5,
    },
]

print(f'✅ Rúbrica cargada: {len(RUBRICA)} criterios')
n_criticos = sum(1 for c in RUBRICA if c['critico'])
print(f'   Criterios críticos (*): {n_criticos}')
peso_total = sum(c['peso'] for c in RUBRICA)
print(f'   Peso total base: {peso_total}')

## NODO 10a — Cargar Qwen3-8B en 4-bit

> ⏱️ Primera carga: ~3-4 minutos (descarga del modelo ~5GB)

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import torch

MODEL_ID = 'Qwen/Qwen3-8B'

# Configuración 4-bit para T4 (16 GB VRAM → ~6 GB para el modelo)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type='nf4',
)

print(f'⏳ Cargando tokenizador {MODEL_ID}...')
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)

print(f'⏳ Cargando modelo en 4-bit...')
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map='auto',
    trust_remote_code=True,
)
model.eval()

vram_usado = torch.cuda.memory_allocated() / 1e9
print(f'✅ Modelo cargado — VRAM usada: {vram_usado:.1f} GB')

# Función de inferencia
def inferir(prompt: str, max_new_tokens: int = 2048, temperatura: float = 0.1) -> str:
    """Ejecuta inferencia con Qwen3-8B y devuelve el texto generado."""
    messages = [
        {'role': 'system', 'content': 'Eres un auditor experto en calidad de llamadas de DIGI Spain Telecom. Analizas llamadas comerciales C2C y devuelves evaluaciones en formato JSON estricto, sin texto adicional.'},
        {'role': 'user', 'content': prompt}
    ]
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False,  # Qwen3 thinking mode OFF → JSON limpio
    )
    inputs = tokenizer([text], return_tensors='pt').to(model.device)
    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=temperatura,
            do_sample=temperatura > 0,
            pad_token_id=tokenizer.eos_token_id,
        )
    # Extraer solo los tokens generados (no el prompt)
    generated = output_ids[0][inputs.input_ids.shape[1]:]
    return tokenizer.decode(generated, skip_special_tokens=True).strip()

print('✅ Función de inferencia lista')

## NODO 10b — Detección del tipo de llamada

In [ ]:
PROMPT_TIPO = f"""Analiza esta transcripción de llamada de DIGI Spain Telecom y determina su tipo.

TRANSCRIPCIÓN:
{transcripcion_raw}

Responde SOLO con este JSON (sin texto antes ni después):
{{
  "tipo_llamada": "NC" | "YC" | "PORT" | "INC" | "INFO",
  "tipo_descripcion": "descripción breve",
  "asesor_speaker": "SPEAKER_00" | "SPEAKER_01" | "desconocido",
  "cliente_speaker": "SPEAKER_00" | "SPEAKER_01" | "desconocido",
  "venta_realizada": true | false,
  "motivo_llamada": "resumen en 1-2 frases"
}}

Tipos: NC=Nuevo Cliente, YC=Ya Cliente, PORT=Portabilidad, INC=Incidencia, INFO=Solo información"""

print('⏳ Detectando tipo de llamada...')
resp_tipo_raw = inferir(PROMPT_TIPO, max_new_tokens=300)
print('Respuesta raw:')
print(resp_tipo_raw)

# Parsear JSON
import re

def extraer_json(texto: str) -> dict:
    """Extrae el primer bloque JSON válido del texto."""
    # Buscar entre llaves
    match = re.search(r'\{.*\}', texto, re.DOTALL)
    if match:
        try:
            return json.loads(match.group())
        except json.JSONDecodeError as e:
            print(f'❌ Error parseando JSON: {e}')
            print(f'   Texto problemático: {match.group()[:200]}')
    return {}

info_llamada = extraer_json(resp_tipo_raw)
print('\n✅ Tipo de llamada detectado:')
print(json.dumps(info_llamada, ensure_ascii=False, indent=2))

## NODO 10c — Análisis completo de la rúbrica

> Prompt único con los 30 criterios → JSON estructurado de auditoría

In [ ]:
# Construir el texto de la rúbrica para el prompt
def rubrica_para_prompt(rubrica: list) -> str:
    lineas = []
    seccion_actual = None
    for c in rubrica:
        if c['seccion'] != seccion_actual:
            seccion_actual = c['seccion']
            lineas.append(f'\n### {seccion_actual}')
        critico_str = ' [CRÍTICO*]' if c['critico'] else ''
        lineas.append(f"""
ID: {c['id']}{critico_str}
Pregunta: {c['pregunta']}
Guía: {c['guia']}
Respuesta válida: Sí / No / N/A""")
    return '\n'.join(lineas)

rubrica_texto = rubrica_para_prompt(RUBRICA)

# Detectar asesor/cliente de la respuesta anterior
asesor_spk = info_llamada.get('asesor_speaker', 'SPEAKER_00')
cliente_spk = info_llamada.get('cliente_speaker', 'SPEAKER_01')
tipo_llamada = info_llamada.get('tipo_llamada', 'NC')

PROMPT_AUDITORIA = f"""Eres un auditor experto de DIGI Spain Telecom. Analiza esta transcripción y evalúa cada criterio de la rúbrica.

INFORMACIÓN DE LA LLAMADA:
- Tipo: {tipo_llamada} ({info_llamada.get('tipo_descripcion', '')})
- Asesor: {asesor_spk}
- Cliente: {cliente_spk}
- Venta realizada: {info_llamada.get('venta_realizada', False)}

TRANSCRIPCIÓN COMPLETA:
{transcripcion_raw}

RÚBRICA DE EVALUACIÓN (30 criterios):
{rubrica_texto}

INSTRUCCIONES:
1. Evalúa SOLO las acciones del ASESOR ({asesor_spk})
2. Para cada criterio: Sí (cumplido), No (incumplido), N/A (no aplica a esta llamada)
3. Usa N/A solo cuando el criterio genuinamente no aplica (ver guía de cada uno)
4. En 'comentario', sé breve (máx 15 palabras) y solo cuando sea No o algo destacable
5. El feedback final (observaciones, puntos_fuertes, areas_mejora) debe ser accionable

Responde ÚNICAMENTE con este JSON (sin texto antes ni después, sin markdown):
{{
  "criterios": [
    {{"id": "p1", "respuesta": "Sí|No|N/A", "comentario": ""}},
    {{"id": "ps1", "respuesta": "Sí|No|N/A", "comentario": ""}},
    {{"id": "ps2", "respuesta": "Sí|No|N/A", "comentario": ""}},
    {{"id": "s1", "respuesta": "Sí|No|N/A", "comentario": ""}},
    {{"id": "s2", "respuesta": "Sí|No|N/A", "comentario": ""}},
    {{"id": "s3", "respuesta": "Sí|No|N/A", "comentario": ""}},
    {{"id": "ao1", "respuesta": "Sí|No|N/A", "comentario": ""}},
    {{"id": "ao2", "respuesta": "Sí|No|N/A", "comentario": ""}},
    {{"id": "ao3", "respuesta": "Sí|No|N/A", "comentario": ""}},
    {{"id": "ao4", "respuesta": "Sí|No|N/A", "comentario": ""}},
    {{"id": "ao5", "respuesta": "Sí|No|N/A", "comentario": ""}},
    {{"id": "cv1", "respuesta": "Sí|No|N/A", "comentario": ""}},
    {{"id": "cv2", "respuesta": "Sí|No|N/A", "comentario": ""}},
    {{"id": "cv3", "respuesta": "Sí|No|N/A", "comentario": ""}},
    {{"id": "td1", "respuesta": "Sí|No|N/A", "comentario": ""}},
    {{"id": "td2", "respuesta": "Sí|No|N/A", "comentario": ""}},
    {{"id": "td3", "respuesta": "Sí|No|N/A", "comentario": ""}},
    {{"id": "cl1", "respuesta": "Sí|No|N/A", "comentario": ""}},
    {{"id": "cl2", "respuesta": "Sí|No|N/A", "comentario": ""}},
    {{"id": "cl3", "respuesta": "Sí|No|N/A", "comentario": ""}},
    {{"id": "tc1", "respuesta": "Sí|No|N/A", "comentario": ""}},
    {{"id": "tc2", "respuesta": "Sí|No|N/A", "comentario": ""}},
    {{"id": "tc3", "respuesta": "Sí|No|N/A", "comentario": ""}},
    {{"id": "tc4", "respuesta": "Sí|No|N/A", "comentario": ""}},
    {{"id": "ge1", "respuesta": "Sí|No|N/A", "comentario": ""}},
    {{"id": "ge2", "respuesta": "Sí|No|N/A", "comentario": ""}},
    {{"id": "cc1", "respuesta": "Sí|No|N/A", "comentario": ""}},
    {{"id": "cc2", "respuesta": "Sí|No|N/A", "comentario": ""}},
    {{"id": "cc3", "respuesta": "Sí|No|N/A", "comentario": ""}},
    {{"id": "cc4", "respuesta": "Sí|No|N/A", "comentario": ""}}
  ],
  "observaciones": "Párrafo de 3-5 frases describiendo el desempeño general del asesor.",
  "puntos_fuertes": ["punto fuerte 1", "punto fuerte 2", "punto fuerte 3"],
  "areas_mejora": ["área mejora 1", "área mejora 2", "área mejora 3"]
}}"""

print('⏳ Analizando llamada contra rúbrica DIGI (puede tardar 1-2 min)...')
resp_auditoria_raw = inferir(PROMPT_AUDITORIA, max_new_tokens=2000, temperatura=0.05)
print('✅ Análisis completado')
print('\nRespuesta raw (primeros 500 chars):')
print(resp_auditoria_raw[:500])

## NODO 11 — Cálculo de puntuaciones dinámicas

Implementa el sistema de pesos de DIGI:  
- N/A → excluido del denominador  
- Nota = Σ(peso_i × Sí_i) / Σ(peso_j × aplicable_j) × 10

In [ ]:
# Parsear respuesta del LLM
auditoria_llm = extraer_json(resp_auditoria_raw)

if not auditoria_llm or 'criterios' not in auditoria_llm:
    print('❌ JSON inválido — mostrando respuesta completa para diagnóstico:')
    print(resp_auditoria_raw)
    raise ValueError('El LLM no generó JSON válido. Revisa la respuesta arriba.')

print(f'✅ JSON parseado: {len(auditoria_llm["criterios"])} criterios evaluados')

# Crear mapa id → criterio de la rúbrica
rubrica_map = {c['id']: c for c in RUBRICA}

# Crear mapa id → evaluación del LLM
eval_map = {e['id']: e for e in auditoria_llm['criterios']}

# ─── Calcular puntuaciones dinámicas ──────────────────────────────
def calcular_puntuaciones(rubrica, eval_map):
    """
    Sistema de puntuación DIGI:
    - Sí → suma el peso al numerador Y al denominador
    - No → suma 0 al numerador, peso al denominador
    - N/A → excluido de ambos (no penaliza ni computa)
    Nota final = (numerador / denominador) × 10
    """
    criterios_calculados = []
    
    for crit in rubrica:
        cid = crit['id']
        eval_item = eval_map.get(cid, {'respuesta': 'N/A', 'comentario': ''})
        respuesta = eval_item.get('respuesta', 'N/A')
        
        # Normalizar respuesta
        respuesta = respuesta.strip().capitalize()
        if respuesta not in ['Sí', 'Si', 'No', 'N/a', 'N/A']:
            print(f'⚠️  Respuesta inválida para {cid}: "{respuesta}" → tratado como N/A')
            respuesta = 'N/A'
        if respuesta in ['Si', 'Sí']:
            respuesta = 'Sí'
        elif respuesta in ['N/a', 'n/a']:
            respuesta = 'N/A'
        
        criterios_calculados.append({
            'id': cid,
            'seccion': crit['seccion'],
            'subseccion': crit['subseccion'],
            'pregunta': crit['pregunta'],
            'critico': crit['critico'],
            'respuesta': respuesta,
            'peso_base': crit['peso'],
            'comentario': eval_item.get('comentario', ''),
        })
    
    return criterios_calculados


def calcular_nota(criterios_calculados, nombre_seccion=None):
    """Calcula la nota de una sección o del total."""
    if nombre_seccion:
        items = [c for c in criterios_calculados if c['seccion'] == nombre_seccion]
    else:
        items = criterios_calculados
    
    denominador = sum(c['peso_base'] for c in items if c['respuesta'] != 'N/A')
    numerador = sum(c['peso_base'] for c in items if c['respuesta'] == 'Sí')
    
    if denominador == 0:
        return 0.0, numerador, denominador
    
    nota = round((numerador / denominador) * 10, 2)
    return nota, numerador, denominador


def asignar_puntuacion_display(criterios_calculados, denominador_total, numerador_total):
    """
    Asigna la puntuación de display (como en los PDFs de DIGI):
    - Escala el peso_base al peso_normalizado que suma 10
    - Sí → muestra puntuación_norm, No → 0, N/A → 0
    """
    items_aplicables = [c for c in criterios_calculados if c['respuesta'] != 'N/A']
    peso_aplicable_total = sum(c['peso_base'] for c in items_aplicables)
    
    for c in criterios_calculados:
        if c['respuesta'] == 'N/A':
            c['puntuacion'] = 0.0
            c['peso_normalizado'] = 0.0
        else:
            # Peso normalizado: escala al espacio disponible (10 puntos - N/A)
            if peso_aplicable_total > 0:
                c['peso_normalizado'] = round(c['peso_base'] / peso_aplicable_total * 10, 2)
            else:
                c['peso_normalizado'] = 0.0
            c['puntuacion'] = c['peso_normalizado'] if c['respuesta'] == 'Sí' else 0.0
    
    return criterios_calculados


# Ejecutar cálculos
criterios_calculados = calcular_puntuaciones(RUBRICA, eval_map)

nota_total, num_total, den_total = calcular_nota(criterios_calculados)
nota_estructura, _, _ = calcular_nota(criterios_calculados, 'Estructura de la llamada')
nota_actitud, _, _ = calcular_nota(criterios_calculados, 'Actitud / Comunicación')

criterios_calculados = asignar_puntuacion_display(criterios_calculados, den_total, num_total)

# Contar respuestas
n_si = sum(1 for c in criterios_calculados if c['respuesta'] == 'Sí')
n_no = sum(1 for c in criterios_calculados if c['respuesta'] == 'No')
n_na = sum(1 for c in criterios_calculados if c['respuesta'] == 'N/A')
n_criticos_fallados = sum(1 for c in criterios_calculados if c['respuesta'] == 'No' and c['critico'])

print('\n' + '=' * 50)
print('📊 RESULTADOS DE AUDITORÍA')
print('=' * 50)
print(f'   Nota final:          {nota_total:.2f} / 10')
print(f'   Estructura llamada:  {nota_estructura:.2f} / 10')
print(f'   Actitud/Comunicación:{nota_actitud:.2f} / 10')
print(f'   Criterios Sí:        {n_si}')
print(f'   Criterios No:        {n_no}')
print(f'   Criterios N/A:       {n_na}')
if n_criticos_fallados:
    print(f'   ⚠️  CRÍTICOS FALLADOS: {n_criticos_fallados}')
print('=' * 50)

## PASO FINAL — Construir y guardar resultado_fase3.json

In [ ]:
from datetime import datetime

# ─── Agrupar criterios por sección para el JSON de salida ─────────
def agrupar_criterios(criterios_calculados):
    secciones = {}
    for c in criterios_calculados:
        sec = c['seccion']
        sub = c['subseccion']
        if sec not in secciones:
            secciones[sec] = {}
        if sub not in secciones[sec]:
            secciones[sec][sub] = []
        secciones[sec][sub].append({
            'id': c['id'],
            'pregunta': c['pregunta'],
            'critico': c['critico'],
            'respuesta': c['respuesta'],
            'puntuacion': c['puntuacion'],
            'peso_normalizado': c['peso_normalizado'],
            'comentario': c['comentario'],
        })
    return secciones


secciones_agrupadas = agrupar_criterios(criterios_calculados)

# Detectar tipificación (tipo de llamada como aparece en los PDFs)
tipo_map = {
    'NC': 'Nuevo Cliente',
    'YC': 'Ya Cliente (gestión)',
    'PORT': 'Portabilidad',
    'INC': 'Incidencia',
    'INFO': 'Consulta / Información',
}
tipificacion = tipo_map.get(tipo_llamada, tipo_llamada)

# ─── JSON final ────────────────────────────────────────────────────
resultado = {
    'version': '1.0',
    'pipeline': 'auditoria-digi-fase3',
    'generado_en': datetime.now().isoformat(),
    # Metadatos de la llamada
    'id_grabacion': META.get('id_grabacion') or 'GRAB-' + datetime.now().strftime('%Y%m%d-%H%M%S'),
    'asesor': META['asesor'],
    'fecha_llamada': META.get('fecha_llamada') or str(date.today()),
    'tipificacion': tipificacion,
    'tipo_codigo': tipo_llamada,
    'fecha_auditoria': META['fecha_auditoria'],
    'servicio': META['servicio'],
    # Puntuaciones
    'nota_final': nota_total,
    'nota_estructura_llamada': nota_estructura,
    'nota_actitud_comunicacion': nota_actitud,
    'estadisticas': {
        'n_si': n_si,
        'n_no': n_no,
        'n_na': n_na,
        'n_criticos_fallados': n_criticos_fallados,
    },
    # Evaluación detallada
    'secciones': secciones_agrupadas,
    # Feedback narrativo
    'observaciones': auditoria_llm.get('observaciones', ''),
    'puntos_fuertes': auditoria_llm.get('puntos_fuertes', []),
    'areas_mejora': auditoria_llm.get('areas_mejora', []),
    # Info técnica
    'modelo_llm': 'Qwen/Qwen3-8B (4-bit, Colab T4)',
    'guion_fuente': str(GUION_PATH),
}

# Guardar
RESULTADO_PATH.parent.mkdir(parents=True, exist_ok=True)
with open(RESULTADO_PATH, 'w', encoding='utf-8') as f:
    json.dump(resultado, f, ensure_ascii=False, indent=2)

print(f'✅ Guardado: {RESULTADO_PATH}')
print(f'   Tamaño: {RESULTADO_PATH.stat().st_size:,} bytes')
print()
print('📋 Resumen del resultado:')
print(f"   ID grabación:  {resultado['id_grabacion']}")
print(f"   Asesor:        {resultado['asesor']}")
print(f"   Tipificación:  {resultado['tipificacion']}")
print(f"   Nota final:    {resultado['nota_final']:.2f} / 10")
print(f"   Estructura:    {resultado['nota_estructura_llamada']:.2f} / 10")
print(f"   Actitud:       {resultado['nota_actitud_comunicacion']:.2f} / 10")
print()
print('🔍 Preview de secciones:')
for seccion, subsecciones in secciones_agrupadas.items():
    n_crit = sum(len(items) for items in subsecciones.values())
    pts = sum(c['puntuacion'] for items in subsecciones.values() for c in items)
    print(f'   {seccion}: {n_crit} criterios, {pts:.2f} pts')

## ✅ Verificación final — Preview del informe

In [ ]:
# Verificar integridad del JSON guardado
with open(RESULTADO_PATH, 'r', encoding='utf-8') as f:
    resultado_verificado = json.load(f)

print('✅ JSON verificado — integridad OK')
print()
print('📊 TABLA DE RESULTADOS (para Fase 4 — PDF):')
print('-' * 80)
print(f'{"ID":<5} {"Pregunta (resumida)":<45} {"Resp":<6} {"Punt":>6}  Critico')
print('-' * 80)

for seccion, subsecciones in resultado_verificado['secciones'].items():
    print(f'\n▶ {seccion}')
    for sub, items in subsecciones.items():
        print(f'  {sub}:')
        for item in items:
            pregunta_corta = item['pregunta'][:42] + '...' if len(item['pregunta']) > 42 else item['pregunta']
            critico_str = ' ⚡' if item['critico'] else ''
            punt_str = f"{item['puntuacion']:.2f}" if item['respuesta'] != 'N/A' else 'N/A'
            print(f"    {item['id']:<5} {pregunta_corta:<45} {item['respuesta']:<6} {punt_str:>6}{critico_str}")

print()
print('─' * 80)
print(f'NOTA FINAL: {resultado_verificado["nota_final"]:.2f} / 10')
print('─' * 80)
print()
print('💬 OBSERVACIONES:')
print(resultado_verificado['observaciones'])
print()
print('✨ PUNTOS FUERTES:')
for p in resultado_verificado['puntos_fuertes']:
    print(f'  • {p}')
print()
print('📈 ÁREAS DE MEJORA:')
for a in resultado_verificado['areas_mejora']:
    print(f'  • {a}')
print()
print('🚀 Fase 3 completada — resultado_fase3.json listo para Fase 4 (PDF)')